## ¿Cómo puedo diagnosticar problemas de descarga?


Hay varias formas de diagnosticar cualquier problema que experimentes al intentar descargar datos desde un servidor `OPeNDAP` mediante `pydap`. Descargar datos a través de `OPeNDAP`/`pydap` significa descargar solo metadatos (es decir, `dmr` o `dds/das`) o datos binarios (`dap` / `dods`) desde un servidor `OPeNDAP`. Puedes descargar distintos tipos de Responses dependiendo del sufijo que agregues a una URL.

| Sufijo agregable | Tipo de Response |
| :- | -: |
| .dmr | Metadatos DAP4 |
| .dmr.html | Formulario de solicitud DAP4 |
| .dmr.xml | Metadatos DAP4 |
| .dds | Metadatos DAP2 |
| .das | Metadatos DAP2 |
| .html | Formulario de solicitud DAP2 |
| .dap | Binario DAP4 |
| .dods | Binario DAP2 |

```{note}
Si usas principalmente `xarray` y la generación del dataset o los tiempos de descarga son lentos, intenta usar solo `pydap`. `xarray` tiende a ser más lento que pydap por toda la funcionalidad extra que agrega al dataset (y muchas comprobaciones internas necesarias para hacerlo).
```
### <font size="5"><span style='color:#0066cc'> **Enfoque Pythonico**<font size="3">
`pydap.client.open_url` usa la biblioteca [requests](https://requests.readthedocs.io/en/latest/) de Python para autenticarse y descargar datos. Puedes intentar descargar cualquiera de las siguientes respuestas mediante `requests.session`:

```python
import requests
session = requests.session()

# assuming url points to a DAP4 dataset, otherwise replace `dmr` with `dds` and `dap` with `dods`
rdmr = session.get(data_url+".dmr")
rdap = session.get(data_url+".dap")
```
`a)` Si `rdmr` devuelve un código de estado `200`, entonces `pydap` debería poder crear un dataset pydap correctamente. Si `rdmr` devuelve un error HTTP [401] o [403], es posible que tengas problemas de autenticación. Asegúrate de tener las credenciales correctas guardadas en un archivo local `netrc` y de que sigan siendo válidas. `requests` y, por tanto, `pydap`, deberían recuperar estas credenciales automáticamente siempre que `netrc` esté en la ubicación predeterminada.
```{warning}
Algunos servidores `GrADS` antiguos exponen una URL DAP2 de datos que comienza con `http://`, aunque este `url-scheme` **ya no** está soportado por NASA. Intenta reemplazar `http` por `https` en la URL. Consulta este [issue de github de 2025](https://github.com/pydap/pydap/issues/460).
```
`b)` Si tanto `rdmr` como `rdap` son mucho más rápidos que `pydap` al crear un dataset o descargar datos, es posible que el dataset contenga una gran cantidad de variables, o que los datos remotos tengan muchos chunks pequeños. Intenta primero crear un dataset `pydap` con solo unas pocas variables del dataset remoto y subdividirlas por sus índices. Esta [documentación](ConstraintExpressions) debería ayudarte a aprender cómo hacerlo.
```{warning}
Si la creación de `dataset` es rápida, pero la descarga del arreglo es extremadamente lenta, entonces es muy probable que las variables del dataset remoto tengan muchos chunks muy pequeños. Una señal de este comportamiento es cuando la descarga de datos binarios es extremadamente lenta comparada con los metadatos. Este escenario es desafortunado. Una cosa que puedes hacer es descargar muchos subconjuntos espaciales del dataset remoto y agregarlos en tu máquina.
```

### <font size="5"><span style='color:#0066cc'> **curl**<font size="3">

`Curl` es una gran herramienta para diagnosticar errores HTTP como problemas de redirección, errores de autenticación, etc. Si no puedes descargar una respuesta OPeNDAP con curl, probablemente tampoco podrás descargarla con pydap.

El siguiente comando es útil al descargar:
```
curl -L -n -v -o output.dmr "http:// ... .dmr"
```

donde `-L` implica seguir redirecciones, `-n` indica a curl que recupere credenciales de autenticación desde el archivo `.netrc` (en la ubicación predeterminada), `-v` indica a `curl` que sea verboso, y `-o` implica descargar el recurso remoto en un archivo llamado `output.dmr`.


Si el tiempo sigue siendo un problema o los errores HTTP son persistentes, considera abrir un issue en [pydap/issue_tracker](https://github.com/pydap/pydap/issues).
